### **Project วิชา Mathematical for Data Science**
> จัดทำโดย นายอรรถกร นาราช 67011578

Modulation คือ กระบวนการนำสัญญาณข้อมูลมาผสมกับคลื่นพาห์ (Carrier Signal) เพื่อสามารถส่งสัญญาณผ่านสายเคเบิลหรือส่งผ่านคลื่นแม่เหล็กไฟฟ้าได้ ปัจจุบันมีเทคนิคการ Modulation มากมาย ยกตัวอย่างเช่น BPSK, QAM ฯลฯ ขึ้นอยู่กับการพัฒนาของเทคโนโลยี ณ เวลานั้น

Project นี้ต้องการนำเสนอการใช้งานความรู้เรื่องของ **Data Science** หรือ **วิทยาการข้อมูล**
นำมาประยุกต์ใช้งานร่วมกับความรู้เรื่อง Digital Communication โดยมีวัตถุประสงค์ดังนี้
- นำเสนอการประยุกต์เอาความรู้ของ Data Science ใช้งานร่วมกับ Principle of Communication ในการจำแนกประเภทของการ Modulation แต่ละประเภทจำแนกเป็น BPSK, QPSK และ QAM16
- สามารถวัดผลของโมเดลด้วยการใช้งาน Accuracy F1-score 
- ฝึกทักษะการใช้งาน Generative AI ร่วมกับการพัฒนาโปรแกรม Python เพื่อสร้างโมเดลทำนาย

โดยในไฟล์ Jupyter Notebook นี้จะประกอบด้วย 2 ส่วน
- การจัดการข้อมูล (Data Manipulation)
- การสร้างโมเดลทำนาย (Model Prediction) 

### การจัดการข้อมูล (Data Manipulation)

แหล่งที่มาของ Dataset : https://www.kaggle.com/datasets/nolasthitnotomorrow/radioml2016-deepsigcom?resource=download

ในการจัดการข้อมูลจะประกอบด้วยการตรวจสอบข้อมูล (Checking), การกรองข้อมูล (Filtering), การรวมข้อมูล (Grouping)

In [2]:
# ทำการ Import Module หรือ Library ที่จำเป็นสำหรับการอ่านข้อมูล
import pickle               # reads .pkl files (Python's format for saving objects)
import zipfile              # opens .zip files
from pathlib import Path    # "Path" makes working with file locations easier
import numpy as np          # numpy = fast number arrays; "np" is a short nickname

# สร้างตัวแปรสำหรับการเก็บ Path ของ Folder ที่มีการเก็บข้อมูล Datasets เอาไว้
data_folder = Path("../data")   # มีการเก็บเอาไว้ใน folder เดียวกันและสร้างเอาไว้ชื่อ data
data_files_name = "radioml-2016-dataset.zip"

# สร้างตัวแปรสำหรับการระบุ Path ของไฟล์ Datasets 
zip_path = data_folder / data_files_name    # ทำการ join ให้อยู่ในรูปแบบ ../data/radioml-2016-dataset.zip
pkl_path = data_folder / "RML2016.10a_dict.pkl" 

# ทำการแสดงข้อความว่า "พบไฟล์" ถ้าไม่พบให้มีการแสดงคำว่า "ไม่พบไฟล์ กรุณาตรวจสอบไฟล์อีกครั้ง"
if(zip_path.exists()==True):
    print("Zip file was found !")
else: 
    print("Zip file doesn't found, Please check your files.")

Zip file was found !


In [3]:
# ขั้นตอนสำหรับการแตกไฟล์ Extract Files เพื่อพร้อมสำหรับการอ่านข้อมูล
# คำสั่ง if not จะมีลักษณะการทำงานคล้าย if-else แต่ว่าจะตรงกันข้าม
if not pkl_path.exists():   # ตรวจสอบว่ามีไฟล์ pkl ไหม ? ให้ทำคำสั่ง else ; ทำคำสั่งด้านล่าง
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_folder)      # ทำการ Unzip ไฟล์เพื่อให้ได้เป็นไฟล์นามสกุล .pkl ออกมา 
    print("Extracted the file.")
else:
    print("Already extracted, skipping.")

# ทำการแสดงผลขนาดของไฟล์ Datasets ออกมาเป็นหน่วย GB
size = pkl_path.stat().st_size  # สร้างตัวแปรสำหรับการเก็บขนาดของ file ซึ่งตรวจสอบออกมาแล้วว่ามีจำนวนเท่ากับ 9 ตัวอักษรตีความว่าจะมีขนาดประมาณ GB
if(len(str(size))>=9):
    print(f"Size : {round(size/1e9,2)} GB") # ถ้าจำนวนตัวอักษรมากกว่าเท่ากับ 9 จะให้มีการแสดงออกมาเป็นหน่วย GB
else:
    print(f"Size : {round(size/1e6,1)} MB") # ถ้าไม่ใช่แสดงว่าน้อยกว่า 9 จะให้มีการแสดงออกมาเป็นหน่วย MB

Already extracted, skipping.
Size : 0.64 GB


In [ ]:
# ทำการตรวจสอบไฟล์ว่าสามารถใช้งานได้หรือไม่
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        allowed = [
            ("numpy", "dtype"),                         
            ("numpy", "ndarray"),                       
            ("numpy.core.multiarray", "_reconstruct"),  
        ]
        if (module, name) in allowed:
            return super().find_class(module, name)     
        raise pickle.UnpicklingError(f"Blocked unsafe item: {module}.{name}")

with open(pkl_path, "rb") as f:
    data = SafeUnpickler(f, encoding="latin1").load()

print("Loaded safely!")

Loaded safely!


In [ ]:
# ทำการตรวจสอบข้อมูล (Checking)

print("Type:", type(data))
print("Number of keys:", len(data))

keys = list(data.keys())
print("First 3 keys:", keys[:3])   # [:3] = the first 3 items

modulations = sorted({k[0] for k in keys})
snrs = sorted({k[1] for k in keys})
print("Modulations:", len(modulations), modulations)
print("SNR levels:", len(snrs), snrs)

one = data[keys[0]]
print("Value type:", type(one))
print("Shape:", one.shape)     
print("Number type:", one.dtype)

print(one[0, :, :5])

Type: <class 'dict'>
Number of keys: 220
First 3 keys: [('QPSK', 2), ('PAM4', 8), ('AM-DSB', -4)]
Modulations: 11 ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']
SNR levels: 20 [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
Value type: <class 'numpy.ndarray'>
Shape: (1000, 2, 128)
Number type: float32
[[-0.00590147 -0.00234582 -0.00074506 -0.00534572 -0.00578942]
 [-0.00779554 -0.00781637 -0.00401967 -0.00511351 -0.00593952]]


In [6]:
# สร้างตัวแปรสำหรับประกาศเป้าหมาย (Targets) ที่ต้องการ Classification
target_mods = ["BPSK", "QPSK", "QAM16"]

## ทำการตรวจสอบว่ามีข้อมูลใน dataset ด้วยการเอา Targets for-loop กับ modulations 
# สร้างตัวแปร m สำหรับการตรวจสอบค่าใน list ของ targets เปรียบเทียบกับ modulations 
missing = [m for m in target_mods if m not in modulations]
assert not missing, f"Not found in the file: {missing}"         # ใช้งาน assert keyword สำหรับการหยุดโปรแกรมในกรณีที่ not missing ถ้าไม่มีข้อมูลใน list จะ return ค่าเป็น True แต่หากมีข้อมูลใน list จะ Return ค่าออกมาเป็น False จึงทำให้ assert ทำงาน

# ทำการดึงข้อมูลจาก datasets ออกมาเก็บเอาไว้ใน list
X_list, y_list, snr_list = [], [], []   # สร้างตัวแปร list 3 ตัวสำหรับการเก็บข้อมูลจาก datasets
for mod in target_mods: # วนลูปแต่ละ Modulation เพื่อเก็บประเภทของข้อมูลและ SNR
    for snr in snrs:
        signals = data[(mod, snr)]  # สร้างตัวแปรสำหรับการเก็บสัญญาณของข้อมูล
        X_list.append(signals)  # ทำการ append เพิ่มข้อมูลลง list ของ X_list โดยที่ X จะเรียกว่า "features"                 
        y_list.append(np.full(len(signals), mod))   # ทำในลักษณะเดียวกันกับข้อมูลที่ได้ทำในบรรทัดก่อนหน้านี้
        snr_list.append(np.full(len(signals), snr)) # ทำในลักษณะเดียวกันกับข้อมูลที่ได้ทำในบรรทัดก่อนหน้านี้

# ใช้งานคำสั่ง np.concatenate เพื่อให้ข้อมูลที่มีการแยกออกจากกันอยู่ด้วยกันเป็นลักษณะคล้าย list ตัวเดียว โดยจะทำกับทั้ง 3 lists
X = np.concatenate(X_list)          # (60000, 2, 128)
y = np.concatenate(y_list)          # (60000,)
snr_arr = np.concatenate(snr_list)  # (60000,)

print("X:", X.shape, "| y:", y.shape, "| snr:", snr_arr.shape)

X: (60000, 2, 128) | y: (60000,) | snr: (60000,)


In [7]:
# ทำการตรวจสอบว่าในทุกการ Modulation และทุก SNR มีจำนวนเท่ากับ 1000 ไหม ? ด้วยการใช้วิธีการนับแบบ Cross-Tabulation
import pandas as pd

counts = pd.crosstab(pd.Series(y, name="modulation"), pd.Series(snr_arr, name="snr"))
print(counts)
print("Every pair has 1,000 signals:", (counts.values == 1000).all())

snr          -20   -18   -16   -14   -12   -10   -8    -6    -4    -2     0   \
modulation                                                                     
BPSK        1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
QAM16       1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   
QPSK        1000  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000   

snr           2     4     6     8     10    12    14    16    18  
modulation                                                        
BPSK        1000  1000  1000  1000  1000  1000  1000  1000  1000  
QAM16       1000  1000  1000  1000  1000  1000  1000  1000  1000  
QPSK        1000  1000  1000  1000  1000  1000  1000  1000  1000  
Every pair has 1,000 signals: True


In [8]:
# ในแต่ละ Feature จะประกอบด้วยการ Modulation สัญญาณของ I และ Q เป็นการ Modulation Techniques
# เราต้องการให้ทั้ง I และ Q อยู่ภายในแถวเดียวกันก็เลยทำการนำเอาสัญญาณของ Q มาต่อ I 
X_flat = X.reshape(len(X), -1)
print("Feature A shape:", X_flat.shape)   # ข้อมูล Features จะเปลี่ยนจาก 2 แถวเหลือเพียงแถวเดียว

Feature A shape: (60000, 256)


In [ ]:
''' 
    ขั้นที่ 3 : ทำการแบ่งข้อมูลออกเป็น train set และ test set
    โดยจะทำการแบ่งข้อมูลออกเป็น 2 กลุ่ม Train set และ Test set 
    Train set = กลุ่มข้อมูลเอาไว้สำหรับการเรียนรู้
    Test set = กลุ่มข้อมูลเอาไว้สำหรับการทดสอบ

    และจะทำการ save เอาไว้ สำหรับการใช้งานอีกครั้ง
'''

from sklearn.model_selection import train_test_split

results_folder = Path("../results")
results_folder.mkdir(exist_ok=True) # exist_ok=True หมายความว่า ถ้ามี folder นี้อยู่แล้ว ไม่ต้องแจ้งเตือน Error
split_path = results_folder / "split_indices.npz"   # สร้างไฟล์ข้อมูลที่ได้มีการแบ่งแล้วชื่อ "split_indices.npz"

if split_path.exists():
    # ถ้ามีไฟล์ split_indices.npz แล้วให้ทำการโหลดไฟล์นั้นแล้วแบ่งออกเป็น Train set และ Test set
    split = np.load(split_path)
    train_idx, test_idx = split["train_idx"], split["test_idx"]
    print("Loaded saved split.")
else:
    # ทำการ split ข้อมูลออกเป็น Train set และ Test set แล้วทำการ save เอาไว้
    strata = np.array([f"{m}_{s}" for m, s in zip(y, snr_arr)])
    all_idx = np.arange(len(y))              # 0, 1, 2, ..., 59999
    train_idx, test_idx = train_test_split(
        all_idx, test_size=0.2, stratify=strata, random_state=42)
    np.savez(split_path, train_idx=train_idx, test_idx=test_idx)
    print("Made a new split and saved it.")

print("Train:", len(train_idx), "| Test:", len(test_idx))   # 48000 | 12000

# ทำการตรวจสอบว่าข้อมูลของ Test set ทุกคู่มีข้อมูลคู่กันระหว่าง Modulation Types และ SNR 
test_counts = pd.crosstab(y[test_idx], snr_arr[test_idx])
print("Every pair has 200 test signals:", (test_counts.values == 200).all())

Made a new split and saved it.
Train: 48000 | Test: 12000
Every pair has 200 test signals: True


In [ ]:
'''
    ขั้นตอนที่ 4 : ทำการเลือก model สำหรับการทำนาย 
    โดยใช้ model การทำนายแบบ Decision Tree และ Gaussian Naive-Bayes 
'''
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

X_train, X_test = X_flat[train_idx], X_flat[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
snr_test = snr_arr[test_idx]
high_snr = snr_test >= 0          # True/False mask: which test signals have SNR >= 0 dB

models = {
    "Decision Tree (max_depth=10, not tuned)": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)          # learn from the train set
    y_pred = model.predict(X_test)       # guess the labels of the test set
    seconds = time.time() - start

    print(f"=== {name} ({seconds:.1f} s) ===")
    print(f"Accuracy (all SNR):  {accuracy_score(y_test, y_pred):.3f}")
    print(f"Macro-F1 (all SNR):  {f1_score(y_test, y_pred, average='macro'):.3f}")
    print(f"Accuracy (SNR >= 0): {accuracy_score(y_test[high_snr], y_pred[high_snr]):.3f}")
    # Rows = true label, columns = predicted label, order = target_mods
    cm = confusion_matrix(y_test[high_snr], y_pred[high_snr], labels=target_mods)
    print(pd.DataFrame(cm, index=target_mods, columns=target_mods))
    print()

=== Decision Tree (max_depth=10, not tuned) (22.0 s) ===
Accuracy (all SNR):  0.425
Macro-F1 (all SNR):  0.404
Accuracy (SNR >= 0): 0.503
       BPSK  QPSK  QAM16
BPSK    774   955    271
QPSK    127  1595    278
QAM16   189  1159    652

=== Gaussian Naive Bayes (0.3 s) ===
Accuracy (all SNR):  0.413
Macro-F1 (all SNR):  0.412
Accuracy (SNR >= 0): 0.481
       BPSK  QPSK  QAM16
BPSK   1205   466    329
QPSK    498  1075    427
QAM16   637   754    609

